# Sepsis Early Warning System
## Notebook 1: Data Extraction

**Author:** Leonardo Alvarino  
**Dataset:** PhysioNet Computing in Cardiology Challenge 2019  
**Source:** https://physionet.org/content/challenge-2019/1.0.0/

---

### Overview
This notebook handles the **Extract** step of the ETL pipeline. It:
1. Downloads the raw PhysioNet 2019 dataset (20,336 ICU patient files)
2. Loads all `.psv` files into a single consolidated DataFrame
3. Saves the result as `full_dataset.csv` for use in downstream notebooks

Each patient file contains hourly ICU readings of vital signs and lab values. The final column `SepsisLabel` indicates sepsis onset (1) or no sepsis (0) at each hour.

> **Note:** Labels are shifted 6 hours ahead per the challenge specification — meaning `SepsisLabel=1` appears 6 hours before clinical sepsis recognition.

## Step 1: Mount Google Drive


In [ ]:
# Mount Google Drive to save our work
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2: Download Dataset from PhysioNet

Credentials are stored securely in Colab Secrets (not hardcoded). To run this notebook:
1. Click the 🔑 key icon in the Colab left sidebar
2. Add a secret named `PHYSIONET_PASSWORD` with your PhysioNet password
3. Enable notebook access for the secret


In [ ]:
import subprocess
import os
from google.colab import userdata

# Create data folder
os.makedirs('/content/drive/MyDrive/DS499-Senior-Project/data', exist_ok=True)

# Credentials stored in Colab Secrets
username = "leonardoalva98"
password = userdata.get('PHYSIONET_PASSWORD')

command = [
    'wget', '-r', '-N', '-c', '-np',
    f'--user={username}',
    f'--password={password}',
    'https://physionet.org/files/challenge-2019/1.0.0/training/training_setA/',
    '-P', '/content/drive/MyDrive/DS499-Senior-Project/data/'
]

result = subprocess.run(command, capture_output=False, text=True)
print("Download complete. Return code:", result.returncode)

KeyboardInterrupt: 

## Step 3: Verify Dataset Files

In [ ]:
import os
import pandas as pd

# Point to your data folder
data_path = '/content/drive/MyDrive/DS499-Senior-Project/data/physionet.org/files/challenge-2019/1.0.0/training/training_setA/'

# List first 5 files to confirm we're in the right place
files = os.listdir(data_path)
print(f"Total patients: {len(files)}")
print("Sample files:", files[:5])

Total patients: 20337
Sample files: ['p019363.psv', 'p019354.psv', 'p019368.psv', 'p019367.psv', 'p019341.psv']


## Step 4: Inspect a Single Patient File

Each `.psv` file contains one row per ICU hour for a single patient. Columns include:
- **Vital signs:** HR, O2Sat, Temp, SBP, MAP, DBP, Resp
- **Lab values:** WBC, Lactate, Creatinine, Glucose, etc.
- **Demographics:** Age, Gender
- **Time:** ICULOS (hours since ICU admission)
- **Target:** SepsisLabel (0 = no sepsis, 1 = sepsis onset)

In [ ]:
# Open one patient file and look at it
df_sample = pd.read_csv(data_path + 'p000001.psv', sep='|')

print(df_sample.shape)  # rows = hours in ICU, columns = features
print(df_sample.head(10))

(54, 41)
      HR  O2Sat   Temp    SBP    MAP  DBP  Resp  EtCO2  BaseExcess  HCO3  ...  \
0    NaN    NaN    NaN    NaN    NaN  NaN   NaN    NaN         NaN   NaN  ...   
1   97.0   95.0    NaN   98.0  75.33  NaN  19.0    NaN         NaN   NaN  ...   
2   89.0   99.0    NaN  122.0  86.00  NaN  22.0    NaN         NaN   NaN  ...   
3   90.0   95.0    NaN    NaN    NaN  NaN  30.0    NaN        24.0   NaN  ...   
4  103.0   88.5    NaN  122.0  91.33  NaN  24.5    NaN         NaN   NaN  ...   
5  110.0   91.0    NaN    NaN    NaN  NaN  22.0    NaN         NaN   NaN  ...   
6  108.0   92.0  36.11  123.0  77.00  NaN  29.0    NaN         NaN   NaN  ...   
7  106.0   90.5    NaN   93.0  76.33  NaN  29.0    NaN         NaN   NaN  ...   
8  104.0   95.0    NaN  133.0  88.33  NaN  26.0    NaN         NaN   NaN  ...   
9  102.0   91.0    NaN  134.0  87.33  NaN  30.0    NaN         NaN   NaN  ...   

   WBC  Fibrinogen  Platelets    Age  Gender  Unit1  Unit2  HospAdmTime  \
0  NaN         NaN      

## Step 5: Check Sepsis Rate Across All Patients

The dataset is heavily imbalanced — most ICU patients do not develop sepsis.
This class imbalance will need to be addressed during model training using `scale_pos_weight`.

In [ ]:
import glob

all_files = glob.glob(data_path + 'p*.psv')

sepsis_count = 0
total_count = 0

for f in all_files:
    df = pd.read_csv(f, sep='|')
    if df['SepsisLabel'].max() == 1:
        sepsis_count += 1
    total_count += 1

print(f"Total patients: {total_count}")
print(f"Sepsis patients: {sepsis_count}")
print(f"Non-sepsis patients: {total_count - sepsis_count}")
print(f"Sepsis rate: {sepsis_count/total_count*100:.1f}%")

Total patients: 20336
Sepsis patients: 1790
Non-sepsis patients: 18546
Sepsis rate: 8.8%


## Step 6: Consolidate All Patient Files into One DataFrame and Save

We loop through all 20,336 patient files and concatenate them into a single DataFrame.
A `patient_id` column is added to preserve patient identity across rows.

> This is the **Extract** step of the ETL pipeline. The result is saved as `full_dataset.csv`
> so subsequent notebooks can load it directly without reprocessing all files.


Save the full dataset to Google Drive. This file is the starting point for all downstream notebooks.

In [ ]:
import pandas as pd
import glob

all_files = glob.glob(data_path + 'p*.psv')

# Load all patients into one dataframe
dfs = []
for f in all_files:
    df = pd.read_csv(f, sep='|')
    patient_id = f.split('/')[-1].replace('.psv', '')
    df['patient_id'] = patient_id
    dfs.append(df)

full_df = pd.concat(dfs, ignore_index=True)
print(f"Full dataset shape: {full_df.shape}")
print(f"Columns: {list(full_df.columns)}")

# Save to CSV
save_path = '/content/drive/MyDrive/DS499-Senior-Project/data/full_dataset.csv'
full_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

Full dataset shape: (790215, 42)
Columns: ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS', 'SepsisLabel', 'patient_id']
Saved to /content/drive/MyDrive/DS499-Senior-Project/data/full_dataset.csv
